In [1]:
import openai

import json
import numpy as np
import os

from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer

from huggingface_hub import InferenceClient

/home/spellen/miniconda3/envs/apm_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load the dataset
with open('ocpq_dataset.json', 'r') as f:
    dataset = json.load(f)

# Load the generated embeddings
embeddings = np.load('embeddings.npy')

# Load the local embedding model so we can embed user queries on the fly
model = SentenceTransformer('./local-all-MiniLM-L6-v2') 

def get_embedding(text):
    return model.encode(text)

# Initialize Hugging Face Inference Client for generation
hf_client = InferenceClient(api_key=os.getenv("HF_TOKEN"))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13967.00it/s]
BertModel LOAD REPORT from: ./local-all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
def retrieve_examples(query, dataset, embeddings, top_k=3):
    """
    Retrieve the top-k most similar examples from the dataset based on cosine similarity of embeddings.
    """
    query_emb = get_embedding(query)
    
    # Calculate cosine similarity between the query and all dataset questions
    similarities = cosine_similarity([query_emb], embeddings)[0]
    
    # Get the indices of the top-k most similar questions
    top_indices = np.argsort(similarities)[-top_k:][::-1]
    
    examples = []
    for idx in top_indices:
        examples.append({
            "nl": dataset[idx]["nl"],
            "nl_clean_1": dataset[idx]["nl_clean_1"] if "nl_clean_1" in dataset[idx] else None,
            "nl_clean_2": dataset[idx]["nl_clean_2"] if "nl_clean_2" in dataset[idx] else None,
            "query": dataset[idx]["query"]
        })
    
    return examples

In [4]:
retrieve_examples("For any person, the diagnosis must be completed within 3 days after the registration at ER.", dataset, embeddings)

[{'nl': 'Ensure that any applications has at least 5 associated candidates where the submit application event has occurred at least 2 times.',
  'nl_clean_1': None,
  'nl_clean_2': None,
  'query': {'nodes': [{'Box': [{'newEventVars': {},
       'newObjectVars': {'0': ['applications']},
       'filters': [],
       'sizeFilters': [],
       'constraints': [{'type': 'SizeFilter',
         'filter': {'type': 'NumChilds',
          'child_name': 'A',
          'min': None,
          'max': 5}}],
       'evVarLabels': {},
       'obVarLabels': {}},
      [1]]},
    {'Box': [{'newEventVars': {'0': ['reject application']},
       'newObjectVars': {'1': ['candidates']},
       'filters': [{'type': 'O2E', 'object': 0, 'event': 0, 'qualifier': None},
        {'type': 'O2E', 'object': 1, 'event': 0, 'qualifier': None}],
       'sizeFilters': [{'type': 'NumChilds',
         'child_name': 'B',
         'min': 2,
         'max': None}],
       'constraints': [],
       'evVarLabels': {},
       'ob

In [5]:
from LogEvents import events_and_objects_from_ocel

event_names, object_names = events_and_objects_from_ocel("../Logs/hospital_log_high.json")

Event names: ['Diagnosis', 'Discharge', 'Examination', 'Expert Examination', 'Register at ER', 'Register at FD', 'Thorough Examination', 'Treatment', 'Treatment successful', 'Treatment unsuccessful']
Object types: ['Person']


In [6]:
import re
import ast

def parse_llm_json(content):
    # Extract json chunk
    match = re.search(r'\{.*\}', content, re.DOTALL)
    clean_content = match.group(0) if match else content.strip()
    
    # Auto-heal truncated JSON
    if not clean_content.endswith("}"):
        clean_content += "\n}"
        
    try:
        return json.loads(clean_content)
    except json.JSONDecodeError:
        try:
            # Fallback: convert python-like string (with single quotes) to Python dict
            py_content = clean_content.replace("null", "None").replace("true", "True").replace("false", "False")
            return ast.literal_eval(py_content)
        except Exception as e:
            print(f"Failed to decode JSON. Error: {e}\nRaw output:\n{content}")
            return None

In [7]:
def nl_to_ocpq_rag(query, dataset, embeddings, query_types=None):
    if query_types is None:
        query_types = ["nl"]
        
    # Retrieve relevant examples
    examples = retrieve_examples(query, dataset, embeddings)
    
    # Construct a prompt with few-shot examples
    example_prompts = []
    for ex in examples:
        for qt in query_types:
            if qt in ex:
                example_prompts.append(
                    f"Question: {ex[qt]}\nJSON: {json.dumps(ex['query'])}"
                )
    few_shot_string = "\n\n".join(example_prompts)
    
    prompt = f"""
Translate natural language process queries into OCPQ JSON based on the examples and the current event and object names.
Output ONLY valid JSON. You MUST use double quotes for all keys and strings.

CRITICAL SCHEMA REQUIREMENT:
The "Box" key MUST always contain an array with exactly TWO elements:
1. The node data dictionary (containing newEventVars, newObjectVars, filters, etc.)
2. An array of child node indices (e.g., [], [1], or [1, 2])
Example: "Box": [ {{ "newEventVars":... }}, [] ]

EXAMPLES:
{few_shot_string}

EVENT NAMES:
{', '.join(event_names)}

OBJECT NAMES:
{', '.join(object_names)}

Now translate this query:
Question: {query}
JSON:
"""

    # Upgrading to a more powerful model for better schema adherence
    response = hf_client.chat_completion(
        model="meta-llama/Meta-Llama-3-70B-Instruct", 
        messages=[
            {"role": "system", "content": "You are a JSON-only API. You must output raw, valid JSON tightly formatted, using double quotes. Strictly follow the OCPQ schema."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=2000,
        response_format={"type": "json_object"}
    )

    content = response.choices[0].message.content
    return parse_llm_json(content)

In [8]:
result = nl_to_ocpq_rag("For any person, the diagnosis must be completed within 3 days after the registration at ER.", dataset, embeddings)
print(json.dumps(result, indent=2))

{
  "nodes": [
    {
      "Box": [
        {
          "newEventVars": {
            "0": [
              "Register at ER"
            ],
            "1": [
              "Diagnosis"
            ]
          },
          "newObjectVars": {
            "0": [
              "Person"
            ]
          },
          "filters": [
            {
              "type": "O2E",
              "object": 0,
              "event": 0,
              "qualifier": null
            },
            {
              "type": "O2E",
              "object": 0,
              "event": 1,
              "qualifier": null
            }
          ],
          "sizeFilters": [],
          "constraints": [
            {
              "type": "TimeConstraint",
              "event1": 0,
              "event2": 1,
              "time_diff": {
                "min": null,
                "max": 3,
                "unit": "days"
              }
            },
            {
              "type": "OrderConstraint",
     

In [9]:
d

NameError: name 'd' is not defined

In [ ]:
result = nl_to_ocpq_rag("write me a poem about shipping order", dataset, embeddings)
print(json.dumps(result, indent=2))

{'nodes': [{'Box': [{'newEventVars': {'0': ['ship package']},
     'newObjectVars': {'0': ['poem']},
     'filters': [],
     'sizeFilters': [],
     'constraints': [],
     'evVarLabels': {},
     'obVarLabels': {}}]},
  []],
 'edgeNames': []}

In [ ]:
result = nl_to_ocpq_rag("no payment reminder should be sent out after payment has been made", dataset, embeddings)
print(json.dumps(result, indent=2))

{'nodes': [{'Box': [{'newEventVars': {'0': ['payment reminder sent out'],
      '1': ['payment has been made']},
     'newObjectVars': {'0': ['communications']},
     'filters': [],
     'sizeFilters': [],
     'constraints': [{'type': 'NOT', 'child_names': ['edge_1']}],
     'evVarLabels': {},
     'obVarLabels': {}}]},
  []],
 'edgeNames': []}

In [ ]:
def nl_to_ocpq(query):

    prompt = f"""
Translate natural language process queries into OCPQ JSON.
Output ONLY valid JSON. You MUST use double quotes for all keys and strings.

CRITICAL SCHEMA REQUIREMENT:
The "Box" key MUST always contain an array with exactly TWO elements:
1. The node data dictionary (containing newEventVars, newObjectVars, filters, etc.)
2. An array of child node indices (e.g., [], [1], or [1, 2])
Example: "Box": [ {{ "newEventVars":... }}, [] ]

Question: {query}
JSON:
"""

    response = hf_client.chat_completion(
        model="meta-llama/Meta-Llama-3-70B-Instruct",
        messages=[
            {"role": "system", "content": "You are a JSON-only API. You must output raw, valid JSON tightly formatted, using double quotes. Strictly follow the OCPQ schema."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=800,
        response_format={"type": "json_object"} 
    )

    content = response.choices[0].message.content
    return parse_llm_json(content)